# Duck Mods v2 — Experiment Notebook (ARC-AGI-3, Kaggle)

**SJSU CMPE 295A Group 2 · July 2026.** A fork of Tufa Labs' milestone-winning
[Duck harness notebook](https://www.kaggle.com/code/jeroencottaar/taaf-duck-harness-kaggle-share)
with a built-in **experiment system**: pick one named preset in section 0, smoke-test with *Save & Run*,
then submit. Full analysis, playbook and the math behind each experiment: `index.html` in our repo.

| Preset | What it tests | Risk |
|---|---|---|
| `stock` | control — identical to Tufa's notebook | none |
| `prompt_only` / `restart_only` | each mod in isolation | low |
| **`mods_v1`** | **recommended: stall restart (27 min/max 4) + prompt addendum** | low |
| `restart_fast` / `restart_slow` / `restart_max` | restart-cadence sweep (18/36/12 min) | low–med |
| `keep_world_model` | restarts that keep the agent's world-model note | low |
| `hot_sampling` / `cold_sampling` | temperature 0.7 / 0.45 | low |
| `long_waves` / `two_waves` | scheduling: 37×170 min / 55×240 min instead of 28×132 | med–high |
| `model_swap_35b` | Qwen3.6-35B-A3B via your own dataset (edit required) | high |
| `custom` | scratch slot | — |

**Setup (once):** attach the competition data + datasets `jeroencottaar/taaf-kaggle-source-share`,
`driessmit1/arc3-vllm-h100-wheelhouse-v3`, `driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot`;
select the **RTX Pro 6000** GPU; internet off. Why these mods exist: identical Duck submissions draw
0.77–1.30 on the LB, and a single pass on the public set has **median ≈ 0.07 vs mean 1.60** — the score
lives in the lucky tail, so we (1) raise EV per run and (2) never skip the daily submission.

## 0. Choose the experiment

One preset per submission. Change one variable at a time, and judge every change over several daily draws — never a single one.

In [ ]:
# ==============================================================================
#  DUCK MODS v2 - EXPERIMENT SELECTOR  (SJSU CMPE 295A Group 2, July 2026)
# ==============================================================================
#  Pick ONE experiment by name in EXPERIMENT below. Every preset is a complete,
#  self-describing configuration; "stock" reproduces Tufa Labs' shared notebook
#  exactly. The engine that applies these settings lives in section 6.
#
#  Workflow:
#    1. Set EXPERIMENT.
#    2. Save & Run  -> offline smoke test on the 25 public games (no submission
#       needed); check the log for "[duck-mods] applied:" lines.
#    3. Submit to the competition (1 submission/day).
#    4. Log the LB score next to the experiment in section 9. Change ONE thing
#       at a time and judge it over SEVERAL daily draws - same-submission
#       scores vary 0.77-1.30, so a single draw proves nothing.

EXPERIMENT = "mods_v1"   # <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<< pick a preset here

# ------------------------------------------------------------------------------
# Baseline = Tufa's stock values. Every preset only overrides what it changes.
# ------------------------------------------------------------------------------
DEFAULTS = {
    "description": "Stock Duck harness (Tufa share notebook, LB ~1.0 median draw).",
    "risk": "none",
    "stall_restart": {
        "enabled": False,        # Mod 1: wipe agent context on stalled games
        "stall_minutes": 27.0,   # minutes without a level-up before a wipe
        "min_session_minutes": 6.0,   # grace period at game start
        "max_restarts": 4,       # per game
        "keep_world_model": False,    # carry the agent's world-model note across wipes?
    },
    "prompt_addendum": False,    # Mod 2: stall-recovery + scoring hints in system prompt
    "sampling": {
        "temperature": None,     # None = stock 0.6
    },
    "solver": {
        "concurrency": None,             # stock: 28 games in parallel
        "max_runtime_s_per_game": None,  # stock: 7920 s (132 min)
        "max_actions_per_game": None,    # stock: unlimited
    },
    "model_swap": {              # EXPERIMENTAL - see section 4 and index.html section 7a
        "enabled": False,
        "model_dataset_source": "YOUR_KAGGLE_USERNAME/qwen3-6-35b-a3b-fp8-snapshot",
        "served_model_name": "local/Qwen3.6-35B-A3B-FP8",
        "max_model_len": 65536,
        "analyzer_context_window": 32768,
    },
}

EXPERIMENTS = {
    # ---- controls -------------------------------------------------------------
    "stock": {
        "description": "Control. Byte-identical behavior to Tufa's shared notebook.",
        "risk": "none",
    },
    "prompt_only": {
        "description": "Isolate Mod 2: prompt addendum only, no restarts.",
        "risk": "low",
        "prompt_addendum": True,
    },
    "restart_only": {
        "description": "Isolate Mod 1: stall restarts only, stock prompt.",
        "risk": "low",
        "stall_restart": {"enabled": True},
    },

    # ---- the recommended config ------------------------------------------------
    "mods_v1": {
        "description": "RECOMMENDED. Stall restart (27 min / max 4) + prompt addendum. "
                       "Turns dead time on stuck games into fresh independent attempts.",
        "risk": "low",
        "stall_restart": {"enabled": True},
        "prompt_addendum": True,
    },

    # ---- restart-cadence sweep (test the core hypothesis) ----------------------
    "restart_fast": {
        "description": "More, shallower attempts: wipe after 18 min, up to 6 per game.",
        "risk": "low",
        "stall_restart": {"enabled": True, "stall_minutes": 18.0,
                          "min_session_minutes": 5.0, "max_restarts": 6},
        "prompt_addendum": True,
    },
    "restart_slow": {
        "description": "Fewer, deeper attempts: wipe after 36 min, up to 3 per game.",
        "risk": "low",
        "stall_restart": {"enabled": True, "stall_minutes": 36.0,
                          "min_session_minutes": 8.0, "max_restarts": 3},
        "prompt_addendum": True,
    },
    "restart_max": {
        "description": "Stress test: wipe every 12 min, up to 8 per game. If this beats "
                       "restart_slow, attempts are nearly independent draws; if it loses "
                       "badly, attempts need depth.",
        "risk": "medium",
        "stall_restart": {"enabled": True, "stall_minutes": 12.0,
                          "min_session_minutes": 4.0, "max_restarts": 8},
        "prompt_addendum": True,
    },
    "keep_world_model": {
        "description": "Like mods_v1 but the world-model note SURVIVES each wipe. Tests "
                       "whether carried memory helps (compounding knowledge) or hurts "
                       "(carried-over wrong theories).",
        "risk": "low",
        "stall_restart": {"enabled": True, "keep_world_model": True},
        "prompt_addendum": True,
    },

    # ---- sampling sweep ---------------------------------------------------------
    "hot_sampling": {
        "description": "mods_v1 + temperature 0.7: more exploration diversity, chasing "
                       "the fat right tail.",
        "risk": "low",
        "stall_restart": {"enabled": True},
        "prompt_addendum": True,
        "sampling": {"temperature": 0.7},
    },
    "cold_sampling": {
        "description": "mods_v1 + temperature 0.45: steadier plans, fewer wild actions "
                       "(efficiency control arm).",
        "risk": "low",
        "stall_restart": {"enabled": True},
        "prompt_addendum": True,
        "sampling": {"temperature": 0.45},
    },

    # ---- scheduling sweep (waves x per-game budget; keep waves*budget <= ~8.6 h) --
    "long_waves": {
        "description": "3 waves of 37 games x 170 min (vs stock 4 x 28 x 132). +29% time "
                       "per game, ~-25% per-game token speed. Restarts at 30 min / max 5.",
        "risk": "medium",
        "stall_restart": {"enabled": True, "stall_minutes": 30.0, "max_restarts": 5},
        "prompt_addendum": True,
        "solver": {"concurrency": 37, "max_runtime_s_per_game": 170 * 60.0},
    },
    "two_waves": {
        "description": "EXTREME: 2 waves of 55 games x 240 min. Heavy batching, slow "
                       "per-game tokens, but each game gets ~2x wall time and up to 7 "
                       "attempts.",
        "risk": "high",
        "stall_restart": {"enabled": True, "stall_minutes": 30.0, "max_restarts": 7},
        "prompt_addendum": True,
        "solver": {"concurrency": 55, "max_runtime_s_per_game": 240 * 60.0},
    },

    # ---- model swap (requires YOUR dataset; edit model_swap fields!) -------------
    "model_swap_35b": {
        "description": "EXPERIMENTAL: serve Qwen3.6-35B-A3B (MoE, ~3-4x faster decode) "
                       "instead of 27B dense. You MUST first snapshot the HF repo into a "
                       "Kaggle dataset, attach it, and edit model_dataset_source below. "
                       "Validate on OpenRouter before burning a submission.",
        "risk": "high",
        "stall_restart": {"enabled": True},
        "prompt_addendum": True,
        "model_swap": {"enabled": True},
    },

    # ---- your scratch config ------------------------------------------------------
    "custom": {
        "description": "Scratch slot - edit freely.",
        "risk": "unknown",
        "stall_restart": {"enabled": True},
        "prompt_addendum": True,
    },
}


def _resolve_config(name: str) -> dict:
    if name not in EXPERIMENTS:
        raise KeyError(f"Unknown EXPERIMENT {name!r}. Choose one of: {sorted(EXPERIMENTS)}")
    import copy as _copy
    cfg = _copy.deepcopy(DEFAULTS)
    for key, value in EXPERIMENTS[name].items():
        if isinstance(value, dict) and isinstance(cfg.get(key), dict):
            cfg[key].update(value)
        else:
            cfg[key] = value
    cfg["experiment"] = name
    return cfg


CONFIG = _resolve_config(EXPERIMENT)

print("=" * 78)
print(f"EXPERIMENT: {EXPERIMENT}   (risk: {CONFIG['risk']})")
print(CONFIG["description"])
print("-" * 78)
_sr = CONFIG["stall_restart"]
print(f"stall_restart : {'ON' if _sr['enabled'] else 'off'}"
      + (f"  ({_sr['stall_minutes']:g} min, warmup {_sr['min_session_minutes']:g} min,"
         f" max {_sr['max_restarts']}, keep_wm={_sr['keep_world_model']})" if _sr["enabled"] else ""))
print(f"prompt_addendum: {'ON' if CONFIG['prompt_addendum'] else 'off'}")
print(f"temperature   : {CONFIG['sampling']['temperature'] or 'stock (0.6)'}")
_so = CONFIG["solver"]
print(f"solver        : concurrency={_so['concurrency'] or 'stock (28)'}, "
      f"per-game budget={_so['max_runtime_s_per_game'] or 'stock (7920 s)'}")
if CONFIG["model_swap"]["enabled"]:
    print(f"MODEL SWAP    : {CONFIG['model_swap']['model_dataset_source']} "
          f"as {CONFIG['model_swap']['served_model_name']}")
    if "YOUR_KAGGLE_USERNAME" in CONFIG["model_swap"]["model_dataset_source"]:
        print("  *** WARNING: placeholder dataset ref - edit model_dataset_source"
              " in DEFAULTS['model_swap'] before running! ***")
print("=" * 78)


## 1. Environment and submission mode

Detect whether this is a real competition rerun (which minimises diagnostics), set the framework's environment flags, and put the CUDA libraries on the linker path.

In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# In submission, disable the periodic JSON/HTML diagnostics writes and per-frame logging.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")

## 2. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment has no internet).

In [ ]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

## 3. Locate the source bundle

Find the uploaded TAAF source dataset by its marker file and record where Kaggle mounted every attached input. If the chosen experiment swaps the model, its dataset is registered here too.

In [ ]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["jeroencottaar/taaf-kaggle-source-share", "driessmit1/arc3-vllm-h100-wheelhouse-v3", "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"]
# DUCK MODS v2: also register the alternative model dataset when the experiment asks for it.
if (CONFIG.get("model_swap") or {}).get("enabled"):
    _swap_ref = CONFIG["model_swap"]["model_dataset_source"]
    if _swap_ref and _swap_ref not in DATASET_SOURCES:
        DATASET_SOURCES.append(_swap_ref)

KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir() -> Path:
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        return marker.parent
    raise RuntimeError("TAAF source bundle not found under /kaggle/input.")


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir()
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

## 4. Import the bundled source and run solver setup

Put the snapshotted repositories on the path, then run the solver's setup commands — installing wheels, starting the vLLM server, and so on. **Model-swap experiments regenerate the vLLM setup command here** (the bundled one is frozen to the 27B model); any failure falls back to the stock model so the run is never lost.

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# Solver setup commands (wheels, vLLM server startup, ...) run before the benchmark loads.
env = _command_env()
_setup_commands = json.loads((BUNDLE_DIR / "setup_commands.json").read_text())

# --- DUCK MODS v2: optional model swap (EXPERIMENTAL, see section 0) -------------
# The bundled setup command was rendered when Tufa built their dataset, so to serve
# a different model we regenerate the vLLM setup command from the bundled source
# (importable at this point) and substitute it. Any failure falls back to stock.
_swap = (CONFIG.get("model_swap") or {})
if _swap.get("enabled"):
    try:
        _found = _first_existing(_dataset_mount_candidates(_swap["model_dataset_source"]))
        if _found is None:
            raise FileNotFoundError(
                f"model dataset not attached: {_swap['model_dataset_source']}"
            )
        import inference.framework.kaggle as _K

        os.environ["LOCAL_ANALYZER_CONTEXT_WINDOW"] = str(_swap.get("analyzer_context_window", 32768))
        _default_model_slug = _K.DEFAULT_QWEN_MODEL_DATASET_SOURCE.split("/", 1)[1]
        _hits = [c for c in _setup_commands if _default_model_slug in c]
        if len(_hits) != 1:
            raise RuntimeError(
                f"expected exactly 1 bundled vLLM setup command, found {len(_hits)} - aborting swap"
            )
        _replacement = _K.duck_kaggle_setup_command(
            _K.DuckKaggleVllmConfig(
                model_dataset_source=_swap["model_dataset_source"],
                served_model_name=_swap["served_model_name"],
                max_model_len=int(_swap.get("max_model_len", 65536)),
            )
        )
        _setup_commands = [
            _replacement if _default_model_slug in c else c for c in _setup_commands
        ]
        print(
            f"duck-mods: MODEL SWAP ACTIVE -> {_swap['model_dataset_source']} "
            f"served as {_swap['served_model_name']}",
            flush=True,
        )
    except Exception as _exc:
        print(f"duck-mods: MODEL SWAP FAILED ({_exc!r}) - falling back to the stock model.", flush=True)

for command in _setup_commands:
    print(f"taaf.kaggle: setup command: {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    # Re-read in case the command persisted new env keys.
    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

## 5. Load the benchmark

Unpickle the deployment target and the benchmark, stamping the real submission state onto the target and pointing the benchmark's outputs at the Kaggle working directory.

In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

## 6. DUCK MODS v2 engine

Applies the chosen CONFIG via defensive in-notebook monkeypatches (the Tufa datasets stay unmodified):

* **Mod 1 — stall restart:** wraps `ToolAgent.analyze`; a game with no level progress for
  `stall_minutes` gets its chat context wiped so a fresh "brain" re-reads the board.
  Completed levels stay banked. Per-game guards: warm-up grace, restart cap.
* **Mod 2 — prompt addendum:** scoring mechanics + "switch hypotheses instead of grinding".
* **Mod 3 — knobs:** temperature, concurrency, per-game budget.
* **Telemetry:** `DUCK_STATS` records restarts and best level per game; section 8 prints it.

If any patch fails it prints the traceback and leaves stock behavior — a mod bug can never cost the daily submission.

In [ ]:
# ==============================================================================
#  DUCK MODS v2 - ENGINE
# ==============================================================================
#  Applies the CONFIG chosen in section 0 to the freshly unpickled benchmark:
#    Mod 1  stall restart  - monkeypatch on ToolAgent.analyze: if a game makes
#           no level progress for stall_minutes, wipe the agent's chat context
#           so a fresh "brain" re-reads the board. Completed levels stay banked
#           (the scorecard keeps them; the game itself is not reset).
#    Mod 2  prompt addendum - appended to the system prompt of every agent.
#    Mod 3  knobs           - temperature (module constant), solver fields.
#
#  Design rules:
#    * The attached Tufa datasets are used UNMODIFIED - everything here is an
#      in-notebook monkeypatch applied after the bundle loads.
#    * Every patch is wrapped defensively: if it fails, stock behavior remains
#      and the failure is printed - a mod bug can never cost the daily
#      submission.
#    * DUCK_STATS collects per-game restart/level telemetry; section 8 prints it.

DUCK_STATS = {"experiment": CONFIG["experiment"], "games": {}, "applied": [], "failed": []}


def _install_duck_mods(cfg, stats):
    import time as _time
    import traceback as _tb

    applied, failed = stats["applied"], stats["failed"]

    try:
        import inference.agent.tool_agent as _ta
    except Exception:
        print("[duck-mods] cannot import tool_agent - no mods applied:\n" + _tb.format_exc(), flush=True)
        return

    # ---- benchmark label (shows up in diagnostics) ----------------------------
    try:
        bm.label = f"{bm.label}-{cfg['experiment']}"
        applied.append("label=" + bm.label)
    except Exception:
        failed.append("label")

    # ---- Mod 2: system-prompt addendum ----------------------------------------
    if cfg.get("prompt_addendum"):
        try:
            _addendum = (
                "\n\nStall recovery and scoring mechanics:\n"
                "- The per-level score is (human_actions / your_actions)^2, so wasted actions are punished "
                "quadratically. Completing a new level is worth far more than polishing an old one, and "
                "levels already completed stay completed even after RESET.\n"
                "- If many turns pass with no level completion, treat your current theory of the game as "
                "probably wrong: write down 2-3 alternative interpretations of the mechanics and run the "
                "cheapest discriminating probe, instead of repeating variations of the same failing plan.\n"
                "- If the level looks unwinnable from the current position, or you have already spent a very "
                "large number of actions on it, RESET the level and execute your best-known route efficiently.\n"
            )
            if hasattr(_ta, "PYTHON_ADDENDUM") and _addendum not in _ta.PYTHON_ADDENDUM:
                _ta.PYTHON_ADDENDUM = _ta.PYTHON_ADDENDUM + _addendum
                applied.append("prompt_addendum")
        except Exception:
            failed.append("prompt_addendum")
            print(_tb.format_exc(), flush=True)

    # ---- Mod 3a: sampling temperature ------------------------------------------
    try:
        _temp = cfg.get("sampling", {}).get("temperature")
        if _temp is not None:
            _ta._LOCAL_ANALYZER_TEMPERATURE = float(_temp)
            applied.append("temperature=%g" % _temp)
    except Exception:
        failed.append("temperature")
        print(_tb.format_exc(), flush=True)

    # ---- Mod 3b: solver knobs ----------------------------------------------------
    for _key, _attr, _cast in (
        ("concurrency", "concurrency", int),
        ("max_runtime_s_per_game", "max_runtime_s_per_game", float),
        ("max_actions_per_game", "max_actions_per_game", int),
    ):
        try:
            _val = cfg.get("solver", {}).get(_key)
            if _val is not None:
                setattr(bm.solver, _attr, _cast(_val))
                applied.append("%s=%s" % (_attr, _val))
        except Exception:
            failed.append(_key)
            print(_tb.format_exc(), flush=True)

    # ---- Mod 1: fresh-context restart on stall ------------------------------------
    _sr = cfg.get("stall_restart", {})
    if _sr.get("enabled"):
        try:
            if getattr(_ta.ToolAgent, "_duck_mods_patched", False):
                applied.append("stall_restart(already patched)")
            else:
                _orig_analyze = _ta.ToolAgent.analyze
                _stall_s = float(_sr["stall_minutes"]) * 60.0
                _warmup_s = float(_sr["min_session_minutes"]) * 60.0
                _max_restarts = int(_sr["max_restarts"])
                _keep_wm = bool(_sr["keep_world_model"])
                _games = stats["games"]

                def _analyze_with_restarts(self, state_path, action_num, **kwargs):
                    key = None
                    try:
                        key = str(state_path.parent)
                        now = _time.monotonic()
                        tr = _games.get(key)
                        if tr is None:
                            tr = _games[key] = {
                                "name": state_path.parent.name,
                                "t0": now, "level": None, "t_prog": now, "restarts": 0,
                            }
                        if (
                            tr["restarts"] < _max_restarts
                            and (now - tr["t0"]) >= _warmup_s
                            and (now - tr["t_prog"]) >= _stall_s
                            and self._history_messages
                        ):
                            self._history_messages = []
                            self._last_step_summary = None
                            if not _keep_wm:
                                try:
                                    self._summarized_knowledge = _ta._empty_world_model()
                                except Exception:
                                    pass
                            tr["restarts"] += 1
                            tr["t_prog"] = now
                            print(
                                "[duck-mods] %s: fresh-context restart #%d after %.0f min without level progress"
                                % (tr["name"], tr["restarts"], _stall_s / 60.0),
                                flush=True,
                            )
                    except Exception:
                        print("[duck-mods] restart check failed:\n" + _tb.format_exc(), flush=True)

                    result = _orig_analyze(self, state_path, action_num, **kwargs)

                    try:
                        lar = getattr(self, "_last_action_result", None) or {}
                        lvl = lar.get("level")
                        tr = _games.get(key) if key is not None else None
                        if tr is not None and isinstance(lvl, int):
                            if tr["level"] is None or lvl > tr["level"]:
                                tr["level"] = lvl
                                tr["t_prog"] = _time.monotonic()
                    except Exception:
                        pass
                    return result

                _ta.ToolAgent.analyze = _analyze_with_restarts
                _ta.ToolAgent._duck_mods_patched = True
                applied.append(
                    "stall_restart(%gmin, warmup %gmin, max %d, keep_wm=%s)"
                    % (_sr["stall_minutes"], _sr["min_session_minutes"], _max_restarts, _keep_wm)
                )
        except Exception:
            failed.append("stall_restart")
            print(_tb.format_exc(), flush=True)

    print("[duck-mods] applied: %s | failed: %s" % (applied or "none", failed or "none"), flush=True)


def _duck_mods_report():
    """Per-game restart/level telemetry - called from the diagnostics section."""
    games = DUCK_STATS["games"]
    print("[duck-mods] experiment=%s | tracked games: %d" % (DUCK_STATS["experiment"], len(games)))
    if not games:
        print("[duck-mods] no per-game telemetry (stall_restart off or no turns played).")
        return
    rows = sorted(games.values(), key=lambda r: (-r["restarts"], r["name"]))
    total_restarts = sum(r["restarts"] for r in rows)
    with_progress = sum(1 for r in rows if (r["level"] or 0) > 1)
    print("[duck-mods] total restarts: %d | games that reached level >= 2: %d" % (total_restarts, with_progress))
    print("  %-34s %9s %11s" % ("game run", "restarts", "best level"))
    for r in rows[:40]:
        print("  %-34s %9d %11s" % (r["name"][:34], r["restarts"], r["level"] if r["level"] is not None else "-"))
    if len(rows) > 40:
        print("  ... (%d more)" % (len(rows) - 40))


_install_duck_mods(CONFIG, DUCK_STATS)


## 7. Run the benchmark

In a real competition rerun, wait for the Kaggle gateway and play the **live competition Arcade**. Otherwise — an interactive "Save & Run" — play the competition's **bundled environment files offline** (25 public games, no gateway), so the notebook runs end-to-end without a submission. Teardown commands run afterward even if the run raises.

In [ ]:
# Build the live competition game list from the gateway's available environments.
def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# Build the offline game list from the competition's bundled environment files.
def _offline_games(env_dir: str):
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    arcade = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# The gateway can take a while to come up; poll until it answers.
def _wait_for_gateway(base_url: str, timeout_s: float = 600.0) -> None:
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")


# Print the run preamble and persist the launcher's git status for diagnostics.
print((BUNDLE_DIR / "preamble.txt").read_text())
(WORKING_DIR / "git_status.txt").write_text((BUNDLE_DIR / "git_status.txt").read_text())

# arc_agi reads RECORDINGS_DIR and ARC_API_KEY from env (ArcadeSpec carries neither); operation
# mode, environments dir, and base url are all passed explicitly via the spec, so no env is needed.
os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

if TRUE_SUBMISSION:
    # Real submission: play the live competition Arcade served by the Kaggle gateway.
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    # The gateway boots asynchronously; wait before swapping in its game list.
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    bm.games = _competition_games()
else:
    # Interactive run: play the bundled competition environments offline (no gateway).
    # The competition's environment files ship alongside the wheelhouse in the competition dataset.
    competition_env_files = str(Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels").parent / "environment_files")
    bm.games = _offline_games(competition_env_files)

bm.n_passes = 1
bm.game_weights = None

# Outside a real submission, stop ~10 min before the wall-clock budget for a graceful exit.
soft_end = None
if not TRUE_SUBMISSION:
    budget = float(getattr(target, "max_runtime_s", 0.0) or 0.0)
    if budget > 0:
        soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(seconds=budget - min(600.0, budget / 2))

# Play the benchmark; teardown commands run even if the run raises.
try:
    await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=TRUE_SUBMISSION)
    if not TRUE_SUBMISSION:
        # An offline run isn't scored, but Kaggle still expects a submission.parquet output.
        import pandas as pd

        pd.DataFrame(
            [["1_0", "1", True, 1]],
            columns=["row_id", "game_id", "end_of_game", "score"],
        ).to_parquet(WORKING_DIR / "submission.parquet", index=False)
finally:
    for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
        print(f"taaf.kaggle: teardown command: {command}", flush=True)
        subprocess.run(command, shell=True, check=False, cwd=WORKING_DIR, env=_command_env())

## 8. Diagnostics + mods telemetry

A non-submission run writes `diagnostics.html` (rendered inline below). The telemetry block prints restarts and best level per game — the quickest way to see whether stall restarts fired and paid off.

In [ ]:
# DUCK MODS v2: per-game restart/level telemetry from this run.
if "_duck_mods_report" in globals():
    _duck_mods_report()

from html import escape

from IPython.display import HTML, display

diagnostics_html = WORKING_DIR / "diagnostics.html"
if diagnostics_html.is_file():
    # Isolate the full document in an iframe so its styles don't leak into the notebook.
    display(
        HTML(
            f'<iframe srcdoc="{escape(diagnostics_html.read_text(), quote=True)}" '
            'width="100%" height="900" style="border:0"></iframe>'
        )
    )
else:
    print("No diagnostics.html — minimal diagnostics (real submission) suppresses it.")

## 9. Experiment log

One row per submission. The public LB is one 50% split of the hidden games (final = the other 50%), so treat every score as a draw from the experiment's distribution — **≥ 3 draws before concluding anything.**

| Date (UTC) | Experiment | LB score | Restarts fired | Notes |
|---|---|---|---|---|
| 2026-07-__ | mods_v1 |  |  |  |
| 2026-07-__ | mods_v1 |  |  |  |
| 2026-07-__ | restart_fast |  |  |  |
| 2026-07-__ | hot_sampling |  |  |  |

**A/B discipline:** cheap multi-seed comparisons belong on the public games via the
[duck-harness GitHub repo](https://github.com/Tufalabs/duck-harness) with OpenRouter
(`CONFIG_PATH=configs/inference.openrouter.json make interactive`) — public per-pass σ ≈ 0.45,
so single-run comparisons are meaningless. Promote a config to Kaggle only when it wins over ≥ 5 seeds.

**Escalation order if plateaued** (details in `index.html` §7): model swap 35B-A3B → prefix-cache-friendly
eviction (needs a source-dataset fork) → perception upgrades (previous-frame image, diff masks).